<a href="https://colab.research.google.com/github/Rojoniaina-Rasoafarihy/Polarization_detection/blob/main/Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount drive



In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

base_path = "/content/drive/MyDrive/PathogenFinder2"

folders = [
    "raw_data/pathogenic/ncbi",
    "raw_data/pathogenic/ena",
    "raw_data/pathogenic/patric"

]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("✅ Folder structure created successfully!")

Mounted at /content/drive
✅ Folder structure created successfully!


## NCBI

In [3]:
df_ncbi = pd.read_csv(
    "/content/drive/MyDrive/PathogenFinder2/raw_data/pathogenic/ncbi/isolates.tsv",
    sep="\t",
    on_bad_lines="skip"
)

df_ncbi.shape

/tmp/ipykernel_13583/3007368114.py:1: DtypeWarning: Columns (3,9,11,12,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ncbi = pd.read_csv(


(2730498, 26)

In [4]:
df_ncbi.drop_duplicates(inplace=True)

In [5]:
df_ncbi["Isolation type"] = (
    df_ncbi["Isolation type"]
    .fillna("non_clinical")
    .str.lower()
    .apply(lambda x: "clinical" if x == "clinical" else "non_clinical")
)

In [6]:
df_ncbi.columns

Index(['#Organism group', 'Strain', 'Isolate identifiers', 'Serovar',
       'Isolate', 'Create date', 'Location', 'Isolation source',
       'Isolation type', 'Food origin', 'Species TaxID', 'Source type',
       'Virulence genotypes', 'K-mer group', 'Host disease', 'Scientific name',
       'Host', 'SNP cluster', 'SRA Center', 'Min-same', 'Min-diff',
       'BioSample', 'TaxID', 'Assembly', 'AMR genotypes', 'Computed types'],
      dtype='object')

In [7]:
selected_columns = [
    "#Organism group", "Assembly", "TaxID", "Scientific name",
    "Host", "Isolation source", "Isolation type",
    "Host disease", "Location"
]
df_ncbi = df_ncbi[selected_columns]
df_ncbi.head()
save_ncbi = "/content/drive/MyDrive/PathogenFinder2/raw_data/pathogenic/ncbi/ncbi.csv"
df_ncbi.to_csv(save_ncbi, index=False)

In [8]:
df_ncbi.shape
df_ncbi.head()

,#Organism group,Assembly,TaxID,Scientific name,Host,Isolation source,Isolation type,Host disease,Location
0,Listeria monocytogenes,GCA_004104335.1,1639,Listeria monocytogenes,NaN,blood,clinical,NaN,USA
1,Listeria monocytogenes,GCA_004104235.1,1639,Listeria monocytogenes,NaN,NaN,clinical,NaN,USA
2,Listeria monocytogenes,GCA_004358545.1,1639,Listeria monocytogenes,NaN,environmental swab,non_clinical,NaN,USA
3,Listeria monocytogenes,GCA_004358555.1,1639,Listeria monocytogenes,NaN,environmental swab,non_clinical,NaN,USA
4,Listeria monocytogenes,GCA_004104175.1,1639,Listeria monocytogenes,NaN,CSF,clinical,NaN,USA


In [9]:
df_ncbi["Isolation type"].value_counts()

,count
Isolation type,
clinical,1789133
non_clinical,941365


### Filter by "pseudomonas aeruginosa" & "blood"

In [10]:
df_ncbi_clinical = df_ncbi[df_ncbi["Isolation type"] == "clinical"]
df_ncbi_clinical.shape

(1789133, 9)

In [11]:
df_ncbi_filter = df_ncbi[ df_ncbi["Scientific name"].str.contains("pseudomonas aeruginosa", case=False, na=False)]
#& df_ncbi["Isolation source"].str.contains("blood", case=False, na=False) & df_ncbi["Host"].str.contains("homo sapiens", case=False, na=False) ]


In [12]:
print(df_ncbi_filter.shape)
df_ncbi_filter.head()

(58217, 9)


,#Organism group,Assembly,TaxID,Scientific name,Host,Isolation source,Isolation type,Host disease,Location
1975399,Pseudomonas aeruginosa,GCA_000006765.1,208964,Pseudomonas aeruginosa PAO1,NaN,NaN,non_clinical,NaN,NaN
1975400,Pseudomonas aeruginosa,GCA_000014625.1,208963,Pseudomonas aeruginosa UCBPP-PA14,NaN,NaN,non_clinical,NaN,NaN
1975401,Pseudomonas aeruginosa,GCA_000026645.1,557722,Pseudomonas aeruginosa LESB58,NaN,NaN,non_clinical,NaN,NaN
1975402,Pseudomonas aeruginosa,GCA_000172395.1,509633,Pseudomonas aeruginosa PAb1,Homo sapiens,clininal isolate from frostbite patient,clinical,NaN,NaN
1975403,Pseudomonas aeruginosa,GCA_000168335.1,388272,Pseudomonas aeruginosa PACS2,NaN,NaN,non_clinical,NaN,NaN


In [13]:
df_ncbi_filter = df_ncbi_filter.drop_duplicates(subset="Assembly")

### Labels of isolation sources

In [14]:
df_ncbi_filter["Isolation source"].value_counts()

,count
Isolation source,
sputum,5833
lung,4515
urine,3144
blood,2306
clinical,1285
...,...
tomato plant,1
cystic fibrosis patient with a chronic infection,1
biofilm from an industrial water system,1


In [15]:
# Get unique values
unique_values = df_ncbi_filter["Isolation source"].dropna().unique()

# Print all
for v in unique_values:
    print(v)

clininal isolate from frostbite patient
cornea of a patient with ulcerative keratitis
gastrointestinal tract
a cystic fibrosis patient
cystic fibrosis chronic infection
soil
air from a glass-crusher air plate
biofilm from an industrial water system
cystic fibrosis patient with a chronic infection
tomato plant
infant with community-acquired diarrhea
sputum sample
blood
sputum of a 14 mo old infant with cystic fibrosis
blood from a hospitalized oncology patient
grown in culture and sent on a short-term space flight
water column
cystic fibrosis isolate
environmental
eye
General Eye
Corneal Scraping
Corneal Scaping
Conjunctiva
Vitreous Fluid
sputum
BAL (mucoid strain)
QC org:ATCC 27853
Lung biopsy
Abd. Woundd
wound
urine
ear
Foot wound
tissue middle turbinate
G-tube site
Groin abscess
shin wound
Abscess (abdominal)
BAL
toenail, subungal debris
Cornea/ocular infection
UTI
Cystic fibrosis
Environmental (water)
Environmental (tomato plant)
Environmental (soil)
patient with pneunomia
Patient (

In [16]:
import pandas as pd


# level 1 and level 2 hierarchical classification
classification = {

    # ---------CLINICAL-------------

    ("Clinical", "Respiratory"): [
        "sputum", "bal", "broncho", "bronchial",
        "trache", "lung", "airway", "respiratory",
        "pharynx", "nasal", "endotracheal",
        "tracheostomy", "lavage", "aspirate",
        "copd", "pneumonia", "ventilator"
    ],

    ("Clinical", "CF_Respiratory"): [
        "cystic fibrosis", "cf sputum",
        "cf respiratory", "cf isolate",
        "cf lung", "cf airway",
        "bronchiectasis"
    ],

    ("Clinical", "Bloodstream"): [
        "blood", "bacteremia",
        "haemoculture", "hemoculture",
        "blood culture", "central line",
        "arterial blood", "venous blood",
        "sepsis"
    ],

    ("Clinical", "Urinary"): [
        "urine", "uti",
        "urinary tract",
        "catheter urine",
        "urinary infection",
        "foley catheter",
        "suprapubic urine",
        "urethra"
    ],

    ("Clinical", "Skin_Soft_Tissue"): [
        "wound", "abscess", "burn",
        "ulcer", "skin", "soft tissue",
        "pus", "lesion", "decubitus",
        "cellulitis", "fasciitis",
        "surgical wound"
    ],

    ("Clinical", "Ocular"): [
        "eye", "cornea", "keratitis",
        "conjunctiva", "vitreous",
        "ocular", "corneal"
    ],

    ("Clinical", "Gastrointestinal"): [
        "stool", "feces", "fecal",
        "rectal", "gut",
        "gastrointestinal", "intestinal",
        "abdomen", "peritoneal",
        "ascitic", "bile",
        "pancreas"
    ],

    ("Clinical", "Ear_Nose_Throat"): [
        "ear", "nose", "throat",
        "sinus", "pharyngeal",
        "nasopharyngeal"
    ],

    ("Clinical", "Reproductive"): [
        "vagina", "vaginal",
        "penis", "uterus",
        "clitoris", "genital",
        "scrotal", "perineal",
        "vulvar"
    ],

    ("Clinical", "Bone_Joint"): [
        "bone", "joint",
        "knee", "hip",
        "tibia", "sacrum",
        "spine", "osteo"
    ],

    ("Clinical", "Medical_Device"): [
        "catheter", "stent",
        "implant", "prosthesis",
        "tube", "line tip",
        "cvc", "cvl"
    ],


    # ---------------ENVIRONMENTAL-------------

    ("Environmental", "Water"): [
        "water", "river", "lake",
        "seawater", "wastewater",
        "ocean", "pond",
        "sink", "drain",
        "tap", "faucet",
        "shower", "spring",
        "well water"
    ],

    ("Environmental", "Soil"): [
        "soil", "sediment",
        "sludge", "rhizosphere",
        "compost", "landfill",
        "mud", "sand"
    ],

    ("Environmental", "Hospital"): [
        "hospital", "icu",
        "ward", "surface",
        "bed rail", "ventilator",
        "room", "hospital sink",
        "medical staff"
    ],

    ("Environmental", "Industrial"): [
        "oil", "fuel tank",
        "wastewater treatment",
        "bioreactor", "petroleum",
        "industrial water",
        "produced fluid"
    ],

    ("Environmental", "Air"): [
        "air", "air plate",
        "air conditioning"
    ],


    # ----------------PLANT---------------

    ("Plant", "Crop"): [
        "tomato", "pepper",
        "lettuce", "onion",
        "potato", "rocket",
        "arugula", "plant",
        "leaf", "root",
        "rhizosphere"
    ],


    # ------------ANIMAL---------------------

    ("Animal", "Domestic"): [
        "dog", "cat",
        "horse", "swine",
        "canine", "feline"
    ],

    ("Animal", "Marine"): [
        "dolphin", "fish",
        "shrimp", "ocean",
        "marine", "turtle"
    ],

    ("Animal", "Bird"): [
        "bird", "cloaca",
        "gallus"
    ],


    # --------------LABORATORY ----------------

    ("Laboratory", "Lab_Strain"): [
        "pao1", "pa14",
        "atcc", "laboratory",
        "lab strain",
        "culture grown",
        "cell culture",
        "chemostat"
    ],

    ("Laboratory", "Experimental"): [
        "mutant", "phage",
        "evolved clone",
        "synthetic cystic fibrosis media",
        "conjugation assay"
    ]
}

In [17]:
# classification function
def classify(text):

    if pd.isna(text):
        return pd.Series(["Unknown", "Unknown"])

    text = str(text).lower()

    for (lvl1, lvl2), keywords in classification.items():

        for kw in keywords:

            if kw in text:
                return pd.Series([lvl1, lvl2])

    return pd.Series(["Unknown", "Unknown"])

In [18]:
df_ncbi_filter[["source_level1", "source_level2"]] = df_ncbi_filter["Isolation source"].apply(classify)

In [19]:
print("\nLEVEL 1 COUNTS\n")
print(df_ncbi_filter["source_level1"].value_counts())

print("\nLEVEL 2 COUNTS\n")
print(df_ncbi_filter["source_level2"].value_counts())


LEVEL 1 COUNTS

source_level1
Clinical         28673
Unknown          17822
Environmental     3369
Laboratory         195
Animal              45
Plant               42
Name: count, dtype: int64

LEVEL 2 COUNTS

source_level2
Unknown             17822
Respiratory         15755
Urinary              3669
Skin_Soft_Tissue     2841
Bloodstream          2593
Hospital             1668
Water                1428
Gastrointestinal     1377
CF_Respiratory        967
Ear_Nose_Throat       917
Ocular                250
Lab_Strain            193
Medical_Device        157
Soil                  134
Air                   123
Bone_Joint             99
Reproductive           48
Crop                   42
Industrial             16
Domestic               16
Marine                 15
Bird                   14
Experimental            2
Name: count, dtype: int64


In [51]:
# create final labeled dataframe
df_labels = df_ncbi_filter[
    [
        "Assembly",
        "Isolation source",
        "source_level1",
        "source_level2"
    ]
].copy()

# preview
df_labels.head()


,Assembly,Isolation source,source_level1,source_level2
1975399,GCA_000006765.1,NaN,Unknown,Unknown
1975400,GCA_000014625.1,NaN,Unknown,Unknown
1975401,GCA_000026645.1,NaN,Unknown,Unknown
1975402,GCA_000172395.1,clininal isolate from frostbite patient,Unknown,Unknown
1975403,GCA_000168335.1,NaN,Unknown,Unknown


In [52]:
# save as CSV
df_labels.to_csv(
    "isolation_source_labels.csv",
    index=False
)

print("CSV saved!")

CSV saved!


## ENA

In [24]:
df_ena = pd.read_csv(
    "/content/drive/MyDrive/PathogenFinder2/raw_data/pathogenic/ena/results_assembly_1.tsv",
    sep="\t"
)

/tmp/ipykernel_13583/2885628364.py:1: DtypeWarning: Columns (1,3,5,6,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ena = pd.read_csv(


In [25]:
df_ena.shape

(4179286, 14)

In [26]:
df_ena.drop_duplicates(inplace=True)

In [27]:
df_ena.head()

,sample_accession,host,assembly_level,sequence_accession,status,assembly_quality,strain,assembly_accession,isolation_source,host_scientific_name,tax_id,host_tax_id,scientific_name,accession
0,SAMN03197995,Homo sapiens,scaffold,NaN,public,NaN,784_LRHA,GCA_001067335,missing,Homo sapiens,47715,9606.0,Lacticaseibacillus rhamnosus,GCA_001067335
1,SAMN03198026,Homo sapiens,scaffold,NaN,public,NaN,812_SBOY,GCA_001067425,missing,Homo sapiens,562,9606.0,Escherichia coli,GCA_001067425
2,SAMN03198028,Homo sapiens,scaffold,NaN,public,NaN,814_SAUR,GCA_001067445,missing,Homo sapiens,1280,9606.0,Staphylococcus aureus,GCA_001067445
3,SAMN03198124,Homo sapiens,contig,NaN,public,NaN,914_NLAC,GCA_001067635,missing,Homo sapiens,267212,9606.0,Neisseria bacilliformis,GCA_001067635
4,SAMN03198132,Homo sapiens,contig,NaN,public,NaN,919_NLAC,GCA_001067655,missing,Homo sapiens,486,9606.0,Neisseria lactamica,GCA_001067655


In [28]:
df_ena["host"].value_counts()

,count
host,
Homo sapiens,1041727
missing,141706
Sus scrofa,26784
Gallus gallus,26221
Bos taurus,19963
...,...
Alnus cremastogyne,1
Rhododendron simsii,1
Gleditsia sinensis,1


In [29]:
df_ena["Isolation type"] = df_ena["host"].apply(
    lambda x: "clinical" if str(x).strip().lower() == "homo sapiens" else "non_clinical"
)

In [30]:
df_ena.columns

Index(['sample_accession', 'host', 'assembly_level', 'sequence_accession',
       'status', 'assembly_quality', 'strain', 'assembly_accession',
       'isolation_source', 'host_scientific_name', 'tax_id', 'host_tax_id',
       'scientific_name', 'accession', 'Isolation type'],
      dtype='object')

In [31]:
selected_columns = [
    "tax_id", "scientific_name", "host_tax_id", "host",
    "host_scientific_name", "isolation_source", "assembly_accession",
    "assembly_level", "status","Isolation type"
]
df_ena = df_ena[selected_columns]
df_ena.head()

save_ena = "/content/drive/MyDrive/PathogenFinder2/raw_data/pathogenic/ncbi/ena.csv"
df_ena.to_csv(save_ena, index=False)

In [32]:
print(df_ena.shape)
df_ena.head()

(4115036, 10)


,tax_id,scientific_name,host_tax_id,host,host_scientific_name,isolation_source,assembly_accession,assembly_level,status,Isolation type
0,47715,Lacticaseibacillus rhamnosus,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067335,scaffold,public,clinical
1,562,Escherichia coli,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067425,scaffold,public,clinical
2,1280,Staphylococcus aureus,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067445,scaffold,public,clinical
3,267212,Neisseria bacilliformis,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067635,contig,public,clinical
4,486,Neisseria lactamica,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067655,contig,public,clinical


In [33]:
df_ena_filter = df_ena[ df_ena["scientific_name"].str.contains("pseudomonas aeruginosa", case=False, na=False)]
#& df_ena["isolation_source"].str.contains("blood", case=False, na=False) & df_ena["host"].str.contains("homo sapiens", case=False, na=False) ]
print(df_ena_filter.shape)
df_ena_filter.head()

(51654, 10)


,tax_id,scientific_name,host_tax_id,host,host_scientific_name,isolation_source,assembly_accession,assembly_level,status,Isolation type
7,287,Pseudomonas aeruginosa,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067725,contig,public,clinical
10,287,Pseudomonas aeruginosa,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067795,contig,public,clinical
15,287,Pseudomonas aeruginosa,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001067965,contig,public,clinical
18,287,Pseudomonas aeruginosa,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001068115,contig,public,clinical
336,287,Pseudomonas aeruginosa,9606.0,Homo sapiens,Homo sapiens,missing,GCA_001076225,scaffold,public,clinical


In [34]:
df_ena_filter["scientific_name"].value_counts()

,count
scientific_name,
Pseudomonas aeruginosa,51209
Pseudomonas aeruginosa PAO1,24
Pseudomonas aeruginosa 7D9A,18
Pseudomonas aeruginosa PA14,15
Pseudomonas aeruginosa 1BAE,14
...,...
Pseudomonas aeruginosa BL18,1
Pseudomonas aeruginosa BL19,1
Pseudomonas aeruginosa BL20,1


In [35]:
df_ena_filter["assembly_accession"].value_counts()

,count
assembly_accession,
GCA_021644425,4
GCA_000215795,3
GCA_002812925,3
GCA_000473745,3
GCA_031480695,3
...,...
GCA_037117315,1
GCA_037117335,1
GCA_037117355,1


In [36]:
df_ena_filter = df_ena_filter.drop_duplicates(subset="assembly_accession")

In [37]:
print(df_ena_filter.shape)

(51275, 10)


## PATRIC

In [38]:
df_patric = pd.read_csv(
    "/content/drive/MyDrive/PathogenFinder2/raw_data/pathogenic/patric/BVBRC_genome.csv"
)

df_patric.shape

/tmp/ipykernel_13583/3900280399.py:1: DtypeWarning: Columns (16,17,18,21,31,32,33,34,37,38,69,76,80,83,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df_patric = pd.read_csv(


(336046, 90)

In [39]:
df_patric.drop_duplicates(inplace=True)

In [40]:
df_patric.head()

,Genome ID,Genome Name,Other Names,NCBI Taxon ID,Taxon Lineage IDs,Taxon Lineage Names,Superkingdom,Kingdom,Phylum,Class,...,Host Age,Host Health,Host Group,Lab Host,Passage,Other Clinical,Additional Metadata,Comments,Date Inserted,Date Modified
0,100053.40,Leptospira alexanderi strain 56650,NaN,100053,131567;2;3379134;203691;203692;1643688;170;171...,cellular organisms;Bacteria;Pseudomonadati;Spi...,Bacteria,Pseudomonadati,Spirochaetota,Spirochaetia,...,NaN,Leptospirosis,Human,NaN,NaN,NaN,collected_by:Jincai Qin,Comparative genomics analysis of 102 Leptospir...,2017-03-20T08:35:16.825Z,2017-03-20T08:35:16.825Z
1,100053.60,Leptospira alexanderi strain 56640,NaN,100053,131567;2;3379134;203691;203692;1643688;170;171...,cellular organisms;Bacteria;Pseudomonadati;Spi...,Bacteria,Pseudomonadati,Spirochaetota,Spirochaetia,...,NaN,Leptospirosis,Human,NaN,NaN,NaN,collected_by:Jincai Qin,Comparative genomics analysis of 102 Leptospir...,2017-03-20T08:35:25.103Z,2017-03-20T08:35:25.103Z
2,100053.80,Leptospira alexanderi strain 56659,NaN,100053,131567;2;3379134;203691;203692;1643688;170;171...,cellular organisms;Bacteria;Pseudomonadati;Spi...,Bacteria,Pseudomonadati,Spirochaetota,Spirochaetia,...,NaN,Leptospirosis,Human,NaN,NaN,NaN,collected_by:Jincai Qin,Comparative genomics analysis of 102 Leptospir...,2017-03-20T08:36:40.695Z,2017-03-20T08:36:40.695Z
3,1000561.30,Pseudomonas aeruginosa AES-1R,NaN,1000561,131567;2;3379134;1224;1236;72274;135621;286;13...,cellular organisms;Bacteria;Pseudomonadati;Pse...,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,...,infant,cystic fibrosis,Human,NaN,NaN,NaN,NaN,Pseudomonas aeruginosa AES-1R was isolated fro...,2014-12-08T22:12:36.342Z,2015-03-16T03:17:09.594Z
4,1000568.21,Megasphaera lornae C043_04,NaN,1000568,131567;2;1783272;1239;909932;1843489;31977;906...,cellular organisms;Bacteria;Bacillati;Bacillot...,Bacteria,Bacillati,Bacillota,Negativicutes,...,NaN,NaN,Human,NaN,NaN,host_subject_id:C043,sample_type:metagenomic assembly,NaN,2025-02-03T22:24:35.692Z,2025-02-03T22:24:35.692Z


In [41]:
df_patric.columns

Index(['Genome ID', 'Genome Name', 'Other Names', 'NCBI Taxon ID',
       'Taxon Lineage IDs', 'Taxon Lineage Names', 'Superkingdom', 'Kingdom',
       'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species',
       'Genome Status', 'Strain', 'Serovar', 'Biovar', 'Pathovar', 'MLST',
       'Segment', 'Subtype', 'H_type', 'N_type', 'H1 Clade Global',
       'H1 Clade US', 'H5 Clade', 'pH1N1-like', 'Lineage', 'Clade', 'Subclade',
       'Other Typing', 'Culture Collection', 'Type Strain', 'Reference',
       'Genome Quality', 'Completion Date', 'Publication', 'Authors',
       'BioProject Accession', 'BioSample Accession', 'Assembly Accession',
       'SRA Accession', 'GenBank Accessions', 'Sequencing Center',
       'Sequencing Status', 'Sequencing Platform', 'Sequencing Depth',
       'Assembly Method', 'Chromosome', 'Plasmids', 'Contigs', 'Size',
       'GC Content', 'Contig L50', 'Contig N50', 'TRNA', 'RRNA', 'Mat Peptide',
       'CDS', 'Coarse Consistency', 'Fine Consistency', 'Ch

In [42]:
selected_columns = [
    "Genome ID", "NCBI Taxon ID", "Species", "Host Name",
    "Host Health", "Isolation Source","Isolation Country",
"Comments", "Other Clinical", "Isolation Comments"
]
df_patric = df_patric[selected_columns]
df_patric.head()
save_patric = "/content/drive/MyDrive/PathogenFinder2/raw_data/pathogenic/ncbi/patric.csv"
df_patric.to_csv(save_patric, index=False)

In [43]:
df_patric_filter = df_patric[ df_patric["Species"].str.contains("pseudomonas aeruginosa", case=False, na=False)]
#& df_patric["Isolation Source"].str.contains("blood", case=False, na=False) ]
print(df_patric_filter.shape)
df_patric_filter.head()

(11758, 10)


,Genome ID,NCBI Taxon ID,Species,Host Name,Host Health,Isolation Source,Isolation Country,Comments,Other Clinical,Isolation Comments
3,1000561.30,1000561,Pseudomonas aeruginosa,"Human, Homo sapiens",cystic fibrosis,the sputum of a 14 month old infant with cysti...,NaN,Pseudomonas aeruginosa AES-1R was isolated fro...,NaN,isolated from the sputum of a 14 month old inf...
1570,1078464.30,1078464,Pseudomonas aeruginosa,"Human, Homo sapiens",NaN,Japan,Japan,MDR P. aeruginosa NCGM1179 was isolated from t...,NaN,MDR P. aeruginosa NCGM1179 was isolated from t...
1585,1081927.11,1081927,Pseudomonas aeruginosa,Homo sapiens,NaN,NaN,NaN,NaN,NaN,NaN
1586,1081927.12,1081927,Pseudomonas aeruginosa,Homo sapiens,NaN,sputum,NaN,NaN,NaN,NaN
1587,1081927.40,1081927,Pseudomonas aeruginosa,"Human, Homo sapiens",NaN,sputum,NaN,Whole-genome sequencing of toxogenic clinical ...,NaN,NaN


In [44]:
df_patric['Other Clinical'].unique()

array([nan, 'host_subject_id:C043', 'host_subject_id:14-2', ...,
       'host_subject_id:Patient 15',
       'host_description:CF patient that regularly attended adult CF clinics',
       'host_description:Pediatric pacient;host_disease_outcome:Death'],
      dtype=object)

In [45]:
df_patric["Isolation type"] = "clinical"

In [46]:
df_patric.head()

,Genome ID,NCBI Taxon ID,Species,Host Name,Host Health,Isolation Source,Isolation Country,Comments,Other Clinical,Isolation Comments,Isolation type
0,100053.40,100053,Leptospira alexanderi,"Human, Homo sapiens",Leptospirosis,NaN,China,Comparative genomics analysis of 102 Leptospir...,NaN,NaN,clinical
1,100053.60,100053,Leptospira alexanderi,"Human, Homo sapiens",Leptospirosis,NaN,China,Comparative genomics analysis of 102 Leptospir...,NaN,NaN,clinical
2,100053.80,100053,Leptospira alexanderi,"Human, Homo sapiens",Leptospirosis,NaN,China,Comparative genomics analysis of 102 Leptospir...,NaN,NaN,clinical
3,1000561.30,1000561,Pseudomonas aeruginosa,"Human, Homo sapiens",cystic fibrosis,the sputum of a 14 month old infant with cysti...,NaN,Pseudomonas aeruginosa AES-1R was isolated fro...,NaN,isolated from the sputum of a 14 month old inf...,clinical
4,1000568.21,1000568,Megasphaera lornae,Homo sapiens,NaN,maternal stool,USA,NaN,host_subject_id:C043,NaN,clinical


In [47]:
df_patric_filter["Species"].value_counts()

,count
Species,
Pseudomonas aeruginosa,11758


In [48]:
df_patric_filter = df_patric_filter.drop_duplicates(subset="Genome ID")

In [49]:
print(df_patric_filter.shape)

(11461, 10)


In [50]:
df_patric_filter["Genome ID"].to_csv("genome_ids.txt", index=False, header=False)